<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_05_lists_tuples_sets/note_lesson_05_lists_tuples_sets_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 5 — Списки, кортежі та множини: нотатка про типи

> Сценарій уроку: кафе закриває день. Треба зберегти позиції замовлень і всі чеки, нічого не переплутати — і відповісти на запитання власниці. Розповідь заняття — у [konspekt_lesson_05_cafe_story](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_05_lists_tuples_sets/konspekt_lesson_05_cafe_story.ipynb), теорія — в [книзі](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_05/).

Ця нотатка — **довідник трьох типів**: усі принципи й методи `list`, `tuple` (+ `NamedTuple`) і `set` з прикладами на даних кафе, типові помилки, шпаргалка, а наприкінці — ті самі ідеї на 244 реальних чеках і задачі з перевірками.

| # | Розділ | Ключове |
|---|---|---|
| 1 | Три контейнери | змінність, порядок, дублікати |
| 2 | Список | індекси, зрізи, усі методи, два імені — один список, `while` |
| 3 | Кортеж | незмінність, пакування/розпакування, хешованість |
| 4 | NamedTuple | `Order` з іменами полів, `_fields` `_asdict` `_replace` |
| 5 | Множина | `set()` ≠ `{}`, усі методи, операції над наборами, `frozenset` |
| 6 | Алгоритми з `while` | пошук, фільтр, reverse, агрегати, злиття |
| 7 | Типові помилки | `IndexError`, `TypeError`, `KeyError`, зміна під час ітерації |
| 8 | Шпаргалка | усі методи в одній таблиці |
| 9 | 244 реальних чеки | той самий код — інші дані |
| 10 | Задачі | з `assert`-перевірками |

Структура: **RETRIEVE → CONCEPT → CREATE → TRANSFER** (та сама, що в уроках 3, 4, 6, 8).


## 🔁 RETRIEVE — пригадай Уроки 3–4 (без підглядання)

1. Що дасть `"Charkiv"[0]` і що — `"Charkiv"[-1]`?
2. Скільки разів виконається блок `while count < 3:`, якщо `count = 0`, а в блоці є `count += 1`?
3. Після `a = 5`, `b = a`, `b = b + 1` — що в `a`? Чому?

<details>
<summary>Відповіді</summary>

1. <code>'C'</code> — індекс 0, <code>'v'</code> — індекс −1. Так само працюватимуть індекси списків і кортежів.
2. Три рази: для 0, 1, 2. На перевірці <code>3 < 3</code> цикл завершується.
3. <code>5</code>: число незмінне, <code>b = b + 1</code> створює нове число і прив'язує до нього ім'я <code>b</code>. Зі списком буде інакше — побачимо в розділі 2.

</details>


## 📖 CONCEPT

### 1. Три контейнери: змінність, порядок, дублікати

Змінна з уроку 4 пам'ятає одне значення. Щоб програма кафе пам'ятала **багато значень**, потрібні контейнери. Головна характеристика, за якою їх розрізняють, — **змінність (mutability)**: чи можна змінити об'єкт після створення.

```text
   LIST  []                 TUPLE  ()                SET  {}
   впорядкований            впорядкований            невпорядкований
   змінний                  незмінний                змінний
   дублікати так            дублікати так            дублікати НІ
   замовлення столика       закритий чек             дні роботи, гості
```


In [ ]:
import sys

raw = ["кава", "чай", "кава", "узвар", "кава"]      # замовлення з повторами

as_list  = list(raw)
as_tuple = tuple(raw)
as_set   = set(raw)

print("list :", as_list)
print("tuple:", as_tuple)
print("set  :", as_set, "  ← дублікати зникли, порядок довільний")
print("Довжини:", len(as_list), len(as_tuple), len(as_set))
print()

as_list[0] = "вода"                                  # список змінюється на місці
print("list після as_list[0] = 'вода':", as_list)
try:
    as_tuple[0] = "вода"
except TypeError as e:
    print("tuple:", e)
try:
    as_set[0]
except TypeError as e:
    print("set  :", e)
print()
print("Пам'ять (байт):", "list", sys.getsizeof(as_list), " tuple", sys.getsizeof(as_tuple), " set", sys.getsizeof(as_set))


| | `list` | `tuple` | `set` |
|---|---|---|---|
| Змінність | змінний | незмінний | змінний |
| Порядок | зберігає | зберігає | не гарантує |
| Дублікати | дозволені | дозволені | видаляються |
| Індекс `x[0]` | так | так | ні |
| Коли обирати | набір, що росте й змінюється | фіксований запис, де кожна позиція має сенс | лише унікальність і швидка перевірка `in` |

Правило вибору — не про синтаксис, а про **сенс**: чи колекція змінюватиметься, чи важливий порядок, чи важлива лише наявність.


### 2. Список — усі принципи й методи

**Список (`list`)** — впорядкована змінна колекція довільних об'єктів. Може містити числа, рядки, інші списки; розмір змінюється під час роботи програми.

#### 2.1 Створення, індекси, зрізи

```text
 items:    ["хліб", "борщ", "вареники", "узвар", "чай"]
 індекс+:      0       1        2         3       4
 індекс−:     -5      -4       -3        -2      -1
```


In [ ]:
items = ["хліб", "борщ", "вареники", "узвар", "чай"]
empty = []                       # порожній список
zeros = [0] * 4                  # повторення: [0, 0, 0, 0]
from_text = list("кава")         # з будь-якої послідовності: ['к', 'а', 'в', 'а']

print(empty, zeros, from_text)
print("len:", len(items), "  перший:", items[0], "  останній:", items[-1])
print()
print("items[1:3]  =", items[1:3], "   від 1 до 3 не включно")
print("items[:2]   =", items[:2])
print("items[2:]   =", items[2:])
print("items[::2]  =", items[::2], "   крок 2")
print("items[::-1] =", items[::-1], "   зворотний порядок")
print("items[1:100]=", items[1:100], "   зріз за межами — не помилка")
print()
copy = items[:]                  # зріз без меж — копія списку
copy[1:3] = ["БОРЩ", "ВАРЕНИКИ"] # присвоєння зрізу замінює кілька елементів
print("після присвоєння зрізу:", copy)
print("оригінал без змін:     ", items)


#### 2.2 Методи, що змінюють список

Кожна подія вечора в кафе — один метод. Усі вони працюють **на місці** і (крім `.pop()`) повертають `None`.


In [ ]:
items = ["борщ", "вареники", "кава"]
print("старт           :", items)

items.append("узвар")             # додати в кінець
print("append('узвар') :", items)

items.insert(0, "хліб")           # вставити на позицію
print("insert(0,'хліб'):", items)

items.extend(["чай", "чай"])      # додати кілька (із будь-якої послідовності)
print("extend([...])   :", items)

items.remove("кава")              # видалити ПЕРШЕ входження значення
print("remove('кава')  :", items)

last = items.pop()                # забрати й повернути останній
print("pop()           :", items, "  → повернув", repr(last))

first = items.pop(0)              # забрати й повернути за індексом
print("pop(0)          :", items, "  → повернув", repr(first))

items[0] = "БОРЩ"                 # замінити за індексом
print("items[0] = ...  :", items)

del items[1]                      # видалити за індексом без повернення
print("del items[1]    :", items)

items.clear()                     # спорожнити
print("clear()         :", items)


#### 2.3 Методи, що читають, і впорядкування

`.count()` та `.index()` нічого не змінюють. Для впорядкування є два шляхи, і їх плутають найчастіше.


In [ ]:
items = ["кава", "чай", "кава", "узвар"]
print("count('кава') =", items.count("кава"))
print("index('чай')  =", items.index("чай"), "   індекс першого входження")
print("'піца' in items =", "піца" in items)
try:
    items.index("піца")
except ValueError as e:
    print("index('піца') → ValueError:", e)
print()

prices = [55, 70, 40, 55]
print("sorted(prices)        =", sorted(prices), "  ← НОВИЙ список")
print("prices                =", prices, "  ← без змін")
print("sorted(..., reverse=True) =", sorted(prices, reverse=True))
prices.sort()                                     # змінює сам список
print("після prices.sort()   =", prices)
prices.reverse()                                  # розгортає на місці
print("після prices.reverse()=", prices)
print("sum / min / max       =", sum(prices), min(prices), max(prices))
print()
oops = [3, 1, 2]
oops = oops.sort()
print("oops = oops.sort() →", oops, "  ← пастка: .sort() повертає None, список втрачено")


#### 2.4 Два імені — один список

В уроці 3 `b = a` для чисел було непомітним, бо числа незмінні. Зі списком це стає головним джерелом загадкових помилок.

🔮 **Передбач:** що надрукує `print(items)` після `kitchen.append("хліб")`?


In [ ]:
items = ["борщ", "вареники"]
kitchen = items                 # друге ім'я ТОГО САМОГО списку
kitchen.append("хліб")
print("items   =", items, "   kitchen is items:", kitchen is items)

backup = items.copy()           # окремий список (те саме: items[:] або list(items))
backup.append("узвар")
print("items   =", items)
print("backup  =", backup, "   backup is items:", backup is items)

# Вкладені списки: меню дня як «столики × позиції»
tables = [
    ["борщ", "хліб"],           # стіл 1
    ["кава"],                   # стіл 2
    ["вареники", "узвар", "чай"],
]
print()
print("tables[2]    =", tables[2])
print("tables[2][1] =", tables[2][1])
print("tables[-1][-1] =", tables[-1][-1])


```text
  items   ──┐
             ├──►  ['борщ', 'вареники', 'хліб']            один об'єкт, два імені
  kitchen ──┘
  backup  ─────►  ['борщ', 'вареники', 'хліб', 'узвар']    .copy() → окремий об'єкт
```

#### 2.5 Прохід списком циклом `while`

Поки що в нас є лише `while` (урок 4): індекс `i` від 0 до `len(items) - 1`. Умова `i < len(items)` гарантує, що індекс не вийде за межі.


In [ ]:
items = ["кава", "чай", "кава"]

coffee = 0
i = 0
while i < len(items):
    if items[i] == "кава":
        coffee += 1
    i += 1                      # без цього — нескінченний цикл

print("Кав у замовленні:", coffee, "   (готовим методом:", items.count("кава"), ")")

# Трейс: що бачить цикл на кожному кроці
i = 0
while i < len(items):
    print(f"  i={i}  items[i]={items[i]!r}")
    i += 1
print(f"  i={i}  {i} < {len(items)} — хибно, цикл завершено")


🔮 **Передбач:** що станеться з `while i <= len(items):`?

<details><summary>Відповідь</summary>
При <code>i = 3</code> умова <code>3 <= 3</code> істинна, а <code>items[3]</code> не існує → <code>IndexError: list index out of range</code>. Та сама помилка «на одиницю», що в уроці 4, але тут вона ламає програму.
</details>

Для «скільки разів» є `.count()`. Але **той самий цикл** потрібен для сум, середніх і максимумів, для яких готового методу немає — розділ 6. В уроці 6 він стане коротшим завдяки `for`.


### 3. Кортеж — усі принципи

**Кортеж (`tuple`)** — впорядкована **незмінна** послідовність. Після створення не можна замінити, додати чи видалити елемент. У кафе — закритий чек: номер столу, сума, чайові.

| Чому кортеж, а не список | Пояснення |
|---|---|
| Безпека | дані захищені від випадкової зміни |
| Сенс позиції | кожна позиція має своє значення: 0 — стіл, 1 — сума, 2 — чайові |
| Продуктивність | менше пам'яті, швидше створення |
| Хешованість | може бути ключем словника чи елементом множини (список — ні) |
| Розпакування | `table, total, tip = receipt` |


In [ ]:
receipt = (4, 860.0, 90.0)          # (стіл, сума, чайові)
empty = ()
single = (42,)                      # ОБОВ'ЯЗКОВА кома
not_tuple = (42)                    # це просто число в дужках
packed = 4, 860.0, 90.0             # без дужок — теж кортеж (пакування)
from_list = tuple(["пт", "сб"])

print(receipt, empty, single, type(not_tuple).__name__, packed, from_list)
print()
print("receipt[0] =", receipt[0], "  receipt[-1] =", receipt[-1], "  receipt[1:] =", receipt[1:])
print("len =", len(receipt), "  860.0 in receipt:", 860.0 in receipt)
print("Методи (єдині два): count(90.0) =", receipt.count(90.0), "  index(860.0) =", receipt.index(860.0))
print()
try:
    receipt[1] = 600.0
except TypeError as e:
    print("TypeError:", e)
print("Немає .append:", hasattr(receipt, "append"))


#### 3.1 Пакування і розпакування


In [ ]:
receipt = (4, 860.0, 90.0)

table, total, tip = receipt                   # кількість змінних = кількість елементів
print(f"стіл {table}: {total} + {tip} = {total + tip} грн до сплати")

first, *rest = (1, 2, 3, 4, 5)                # зірочка збирає «решту» у список
*head, last = (1, 2, 3, 4, 5)
a, *middle, b = (1, 2, 3, 4, 5)
print(first, rest, "|", head, last, "|", a, middle, b)

x, y = 1, 2
x, y = y, x                                   # обмін без тимчасової змінної
print("після обміну:", x, y)

try:
    a, b = receipt
except ValueError as e:
    print("ValueError:", e)


#### 3.2 Незмінність є «відносною», хешованість — абсолютною


In [ ]:
t = (4, ["борщ", "кава"])          # кортеж містить список
t[1].append("чай")                 # список усередині — можна змінити
print(t, "  ← кортеж той самий, змінився вміст списку")
try:
    t[1] = ["піца"]                # замінити посилання — не можна
except TypeError as e:
    print("TypeError:", e)
print()

# Хешованість: кортеж може бути елементом множини, список — ні
tables_served = {(4, "Тарас"), (2, "Марія"), (4, "Тарас")}
print("Множина кортежів:", tables_served)
try:
    {[4, "Тарас"]}
except TypeError as e:
    print("Список у множині → TypeError:", e)

print()
print("Кортежі порівнюються поелементно:", (1, 2) < (1, 3), (4, 860.0) == (4, 860.0))
print("Зріз і конкатенація дають НОВИЙ кортеж:", receipt[:2] + (0.0,))


| | `list` | `tuple` |
|---|---|---|
| Розмір змінюється? | так | ні |
| Що означає позиція? | усі елементи рівноправні | кожна позиція має свій сенс |
| Приклад у кафе | позиції замовлення, чеки дня | один закритий чек |


### 4. NamedTuple — кортеж з іменами полів

**Проблема індексів.** У `receipt = (4, 860.0, 90.0)` треба пам'ятати, що `[1]` — сума, `[2]` — чайові. Коли полів шість, переплутати індекс легко, і Python **не помітить** — мовчки порахує не те.

**`NamedTuple`** дає кожному полю ім'я і лишається звичайним незмінним кортежем: той самий розмір, та сама швидкість, підтримка індексів і розпакування.


In [ ]:
# Старший варіант — з модуля collections (зустрічається в чужому коді)
from collections import namedtuple

Receipt = namedtuple("Receipt", "table total tip")
r = Receipt(4, 860.0, 90.0)
print(r, "→", r.total, r[1])

# Сучасний варіант — typing.NamedTuple з типами полів. Канонічний чек курсу:
from typing import NamedTuple


class Order(NamedTuple):
    waiter: str          # хто обслуговував
    total_bill: float    # сума чека, грн
    tip: float           # чайові, грн
    day: str             # день тижня
    time: str            # "обід" / "вечеря"
    size: int            # гостей за столом


order = Order("Тарас", 540.0, 50.0, "пт", "вечеря", 2)
print()
print(order)
print("Через ім'я:   order.tip =", order.tip)
print("Через індекс: order[2]  =", order[2], " ← те саме")
print("isinstance(order, tuple):", isinstance(order, tuple))

waiter, total_bill, tip, day, time, size = order     # розпакування працює
print(f"{waiter}: {total_bill} грн, {size} гостей, {day} {time}")

try:
    order.tip = 70.0
except AttributeError as e:
    print("AttributeError:", e, " ← незмінний, як і tuple")


Як це читати:

- `from typing import NamedTuple` — підключаємо інструмент з модуля (як `import random` в уроці 4);
- `class Order(NamedTuple):` — **шаблон запису**: назва і список полів; після двокрапки — очікуваний тип;
- `Order("Тарас", 540.0, …)` — створює один чек: значення розкладаються по полях у порядку оголошення.

Слово `class` належить до теми класів (модуль 2); поки що це просто опис полів. Типи — підказка для читача: `Order("п'ятсот", …)` створиться без помилки, а впаде пізніше при обчисленнях.

#### 4.1 Службові методи NamedTuple і значення за замовчуванням


In [ ]:
print("Поля:", Order._fields)
print("Пари ім'я → значення:", order._asdict())

fixed = order._replace(tip=70.0)         # «виправлений» чек — НОВИЙ об'єкт
print("order.tip =", order.tip, "  fixed.tip =", fixed.tip, "  fixed is order:", fixed is order)


class Guest(NamedTuple):
    name: str
    visits: int = 1                      # значення за замовчуванням


print(Guest("Олена"), Guest("Марта", 3))


#### 4.2 Список чеків дня — `list` з `Order`

Один чек — `Order` (незмінний). Усі чеки дня — список (росте протягом дня). Ті самі 5 чеків, що в розповіді, книзі й квесті.


In [ ]:
orders = [
    Order("Тарас",   540.0,  50.0, "пт", "вечеря", 2),
    Order("Марія",   320.0,  30.0, "пт", "обід",   1),
    Order("Тарас",   980.0, 120.0, "сб", "вечеря", 4),
    Order("Олексій", 760.0,  70.0, "сб", "вечеря", 3),
    Order("Марія",   450.0,   0.0, "нд", "обід",   5),
]

print("Чеків:", len(orders), "  третій:", orders[2].total_bill, "грн,", orders[2].waiter)

# Накопичувачі + поточний чемпіон за один прохід
revenue = 0
tips = 0
best = orders[0]
i = 1                                  # перший чек уже стартовий чемпіон, але суми рахуємо з нього
revenue += orders[0].total_bill
tips += orders[0].tip
while i < len(orders):
    order = orders[i]
    revenue += order.total_bill
    tips += order.tip
    if order.total_bill > best.total_bill:
        best = order
    i += 1

print(f"Виторг {revenue} грн, чайові {tips} грн ({tips / revenue * 100:.1f} %)")
print(f"Найбільший чек: {best.total_bill} грн — {best.waiter}")

# Фільтр у НОВИЙ список: великі столи
large = []
i = 0
while i < len(orders):
    if orders[i].size >= 4:
        large.append(orders[i])
    i += 1
print("Великих столів:", len(large), "→", large[0].total_bill, "і", large[1].total_bill, "грн")


### 5. Множина — усі принципи й методи

**Множина (`set`)** — невпорядкована колекція **унікальних** об'єктів на основі хеш-таблиці.

- елементи мають бути **хешованими** (незмінними): числа, рядки, кортежі — так; списки, словники — ні;
- перевірка `in` працює за O(1) — миттєво, незалежно від розміру (список переглядається по одному);
- **порядок не гарантований**, індексів немає;
- **порожня множина — лише `set()`**: `{}` — це порожній словник (урок 6).

#### 5.1 Створення і базові методи


In [ ]:
tables = {3, 1, 2, 3, 1}
from_list = set(["пт", "сб", "пт", "нд", "сб"])
from_text = set("mississippi")
empty = set()
print(tables, from_list, from_text, empty)
print("type({}) =", type({}).__name__, " ← словник, не множина!")
try:
    {["кава", "чай"]}
except TypeError as e:
    print("TypeError:", e)
print()

days = set()
days.add("пт")
days.add("сб")
days.add("пт")                    # уже є — нічого не станеться
print("add     :", days)
print("in      :", "сб" in days, "пн" in days)
days.discard("пн")                # немає — тихо
print("discard :", days)
try:
    days.remove("пн")             # немає — KeyError
except KeyError as e:
    print("remove  : KeyError", e)
print("len     :", len(days), "  sorted →", sorted(days), " (список!)")
copy = days.copy()
popped = days.pop()               # забирає ДОВІЛЬНИЙ елемент
print("pop     :", popped, "лишилось", days, "  копія ціла:", copy)
days.clear()
print("clear   :", days)
try:
    copy[0]
except TypeError as e:
    print("індекс  : TypeError:", e)


#### 5.2 Операції над наборами: гості трьох днів

```text
       friday                 saturday
   ┌─────────────┬───────────┬─────────────┐
   │ Олена       │  Марта    │             │
   │ Богдан      │  Ігор     │  Софія      │
   └─────────────┴───────────┴─────────────┘
      f - s          f & s        s - f
   └──────────────── f | s ────────────────┘
```


In [ ]:
friday   = {"Олена", "Богдан", "Марта", "Ігор"}
saturday = {"Марта", "Ігор", "Софія"}
sunday   = {"Ігор", "Софія", "Петро"}

print("|  union               :", sorted(friday | saturday), "   хто був хоча б раз")
print("&  intersection        :", sorted(friday & saturday), "   хто був обидва дні")
print("-  difference          :", sorted(friday - saturday), "   лише в п'ятницю")
print("^  symmetric_difference:", sorted(friday ^ saturday), "   лише в один з днів")
print()
print("Методи = оператори:", friday.union(saturday) == friday | saturday,
      friday.intersection(saturday) == friday & saturday)
print("Усі три дні:          ", friday & saturday & sunday)
print("Хоча б один день:     ", sorted(friday | saturday | sunday))
print("Лише в п'ятницю:      ", friday - saturday - sunday, "   те саме:", friday.difference(saturday, sunday))
print()
print("issubset   :", {"Марта", "Ігор"}.issubset(friday), "   чи кожен з них був у пт")
print("issuperset :", friday.issuperset({"Марта"}))
print("isdisjoint :", friday.isdisjoint({"Петро"}), "   немає спільних")
print()
regulars = friday.copy()
regulars &= saturday                  # оновлення на місці: &=  |=  -=  ^=
regulars.update(["Зоя"])              # додати кілька одразу
print("regulars після &= і update:", sorted(regulars))
print("Метод приймає список:", sorted(friday.union(["Зоя"])))
try:
    friday | ["Зоя"]                  # оператор — лише між множинами
except TypeError as e:
    print("TypeError:", e)


#### 5.3 Унікальні значення з чеків і `frozenset`


In [ ]:
days = set()
waiters = set()
i = 0
while i < len(orders):
    days.add(orders[i].day)
    waiters.add(orders[i].waiter)
    i += 1
print(f"{len(orders)} чеків → днів {len(days)} {sorted(days)}, офіціантів {len(waiters)} {sorted(waiters)}")

# Прибрати дублікати зі списку одним рядком — але порядок губиться
print("list(set(...)):", list(set(["кава", "чай", "кава"])), " ← порядок може змінитись")

# frozenset — незмінна множина: можна покласти в іншу множину
shift_fri = frozenset({"Тарас", "Марія"})
shift_sat = frozenset({"Тарас", "Олексій"})
all_shifts = {shift_fri, shift_sat, frozenset({"Тарас", "Марія"})}
print("Унікальних складів змін:", len(all_shifts))
try:
    shift_fri.add("Софія")
except AttributeError as e:
    print("frozenset.add → AttributeError:", e)


### 6. Алгоритми з циклом `while`

Хоча Python частіше використовує `for` (урок 6), `while` з лічильником показує **механіку індексів** — основу всіх алгоритмів над масивами.

```text
i = 0                     # лічильник від 0
while i < len(data):      # не вийти за межі
    current = data[i]     # елемент за індексом
    ...                   # логіка
    i += 1                # ОБОВ'ЯЗКОВО
```


In [ ]:
# 6.1 Лінійний пошук: на яких позиціях у замовленні стоїть кава?
items = ["кава", "чай", "кава", "узвар", "кава"]
found = []
i = 0
while i < len(items):
    if items[i] == "кава":
        found.append(i)
    i += 1
print("кава на індексах:", found, "   .index дає лише перший:", items.index("кава"))


In [ ]:
# 6.2 Фільтрація у НОВИЙ список — ніколи не видаляй зі списку, по якому ідеш
bills = [540.0, 320.0, 980.0, 760.0, 450.0]

big = []
i = 0
while i < len(bills):
    if bills[i] > 500:
        big.append(bills[i])
    i += 1
print("чеки > 500:", big)

# Антиприклад: видалення під час проходу пропускає елементи
broken = [540.0, 320.0, 980.0, 760.0, 450.0]
i = 0
while i < len(broken):
    if broken[i] > 500:
        broken.remove(broken[i])      # індекси зсунулись — наступний елемент пропущено
    i += 1
print("після 'видалення' > 500:", broken, " ← 760.0 вціліло, це баг")


In [ ]:
# 6.3 Розгортання списку на місці двома індексами
queue = ["хліб", "борщ", "вареники", "узвар", "чай"]
left = 0
right = len(queue) - 1
while left < right:
    queue[left], queue[right] = queue[right], queue[left]   # обмін через розпакування
    left += 1
    right -= 1
print("reverse на місці:", queue)
print("queue[::-1] дає новий список, .reverse() — на місці")


In [ ]:
# 6.4 Агрегати за ОДИН прохід: сума, максимум, мінімум, лічильник
bills = [540.0, 320.0, 980.0, 760.0, 450.0]

total = 0
biggest = bills[0]          # стартуємо з першого елемента, не з 0
smallest = bills[0]
above_500 = 0
i = 0
while i < len(bills):
    current = bills[i]
    total += current
    if current > biggest:
        biggest = current
    if current < smallest:
        smallest = current
    if current > 500:
        above_500 += 1
    i += 1

print(f"сума {total}, середнє {total / len(bills):.1f}, max {biggest}, min {smallest}, > 500: {above_500}")
print("перевірка вбудованими:", sum(bills), max(bills), min(bills))


In [ ]:
# 6.5 Злиття двох відсортованих списків (два індекси) — черги двох кас за часом
cash_1 = [10, 25, 40, 55]
cash_2 = [15, 20, 60]
merged = []
i = 0
j = 0
while i < len(cash_1) and j < len(cash_2):
    if cash_1[i] <= cash_2[j]:
        merged.append(cash_1[i])
        i += 1
    else:
        merged.append(cash_2[j])
        j += 1
while i < len(cash_1):            # дописати залишок
    merged.append(cash_1[i])
    i += 1
while j < len(cash_2):
    merged.append(cash_2[j])
    j += 1
print("merged:", merged, "  правильно:", merged == sorted(cash_1 + cash_2))


### 7. Типові помилки

| Помилка | Причина | Виправлення |
|---|---|---|
| `IndexError: list index out of range` | індекс ≥ `len()` | `while i < len(items)`; останній індекс `len - 1` |
| `TypeError: 'tuple' object does not support item assignment` | зміна кортежу | створи новий кортеж або використай список |
| `TypeError: 'set' object is not subscriptable` | індекс у множини | `sorted(s)[0]` або прохід циклом |
| `TypeError: unhashable type: 'list'` | список у множині | використай кортеж |
| `ValueError: list.remove(x): x not in list` | видалення відсутнього | перевір `if x in items:` |
| `KeyError` при `.remove()` у множини | відсутній елемент | `.discard()` |
| `AttributeError: 'dict' object has no attribute 'add'` | `{}` замість `set()` | `set()` |
| список став `None` | `prices = prices.sort()` | `prices.sort()` окремо або `sorted(prices)` |
| «кортеж» виявився числом | `(42)` | `(42,)` |
| пропущені елементи | видалення під час ітерації | збирай у новий список |


In [ ]:
items = ["кава", "чай"]
try:
    items[2]
except IndexError as e:
    print("1. IndexError:", e, "  (індекси 0..", len(items) - 1, ")")

try:
    (4, 860.0)[0] = 5
except TypeError as e:
    print("2. TypeError:", e)

days = {"пт", "сб"}
try:
    days[0]
except TypeError as e:
    print("3. TypeError:", e)

try:
    items.remove("піца")
except ValueError as e:
    print("4. ValueError:", e)

d = {}
try:
    d.add("пт")
except AttributeError as e:
    print("5. AttributeError:", e, "  ← {} — це dict")

print("6. (42) →", type((42)).__name__, "   (42,) →", type((42,)).__name__)


### 8. Шпаргалка

```python
# LIST — впорядкований, змінний
items = ["кава", "чай"]        # створити;  list("кава"), [0] * 3
items[0]; items[-1]            # індекс
items[1:3]; items[::-1]        # зріз → новий список
"чай" in items; len(items)
items.append(x)                # у кінець
items.insert(i, x)             # на позицію
items.extend([a, b])           # кілька
items.remove(x)                # перше входження (ValueError, якщо немає)
items.pop(); items.pop(i)      # забрати й повернути
del items[i]; items.clear()
items.index(x); items.count(x)
items.sort(); items.reverse()  # на місці, повертають None
sorted(items); items[::-1]     # новий список
items.copy()                   # окремий список (b = a — НЕ копія)
sum(nums); min(nums); max(nums)

# TUPLE — впорядкований, незмінний
receipt = (4, 860.0, 90.0)     # (42,) — з одного елемента
receipt[0]; receipt[1:]; len(receipt)
receipt.count(x); receipt.index(x)     # єдині методи
table, total, tip = receipt            # розпакування; first, *rest = t
a, b = b, a                            # обмін

# NAMEDTUPLE — tuple з іменами полів
from typing import NamedTuple
class Order(NamedTuple):
    waiter: str
    total_bill: float
    tip: float
    day: str
    time: str
    size: int
order = Order("Тарас", 540.0, 50.0, "пт", "вечеря", 2)
order.tip == order[2]; isinstance(order, tuple)
Order._fields; order._asdict(); order._replace(tip=70.0)

# SET — унікальні, без порядку
s = {1, 2}; set(); set(items)  # {} — це dict!
s.add(x); s.discard(x); s.remove(x)   # remove → KeyError, якщо немає
x in s; len(s); sorted(s)
a | b  a & b  a - b  a ^ b     # union intersection difference symmetric_difference
a.issubset(b); a.issuperset(b); a.isdisjoint(b)
a |= b; a &= b; a.update([...])
frozenset(s)                   # незмінна множина

# WHILE — прохід з індексом
i = 0
while i < len(items):
    ...                        # items[i]
    i += 1
```


## 🛠️ CREATE — ті самі ідеї на 244 реальних чеках

Досі чеки ми вводили вручну. У реальному житті вони приходять з POS-системи, бази даних чи файлу. Датасет `tips` (вбудований у `seaborn`) містить **244 реальних чеки** з ресторану: сума (`total_bill`, USD), чайові (`tip`), день (`day`), час (`time`: Lunch / Dinner), гостей за столом (`size`) — ті самі поля, що в нашому `Order`. Колонок `sex` і `smoker` ми тут не використовуємо.

Офіціанта в датасеті **немає**. Приймаємо припущення: на одну зміну (день + час) виходив один офіціант, і призначаємо імена за правилом нижче. Це симуляція, і в тексті ми про це кажемо чесно.

DataFrame — таблиця `pandas`; наш алгоритм працює зі списком `Order`, тож спершу конвертуємо тим самим `while`, що й завжди: `tips_df.iloc[i]` — рядок з індексом `i`.


In [ ]:
import seaborn as sns

tips_df = sns.load_dataset("tips")
print(f"Завантажено: {len(tips_df)} рядків, колонки: {list(tips_df.columns)}")
print(tips_df.head(3).to_string())
print()

orders = []                                   # той самий список Order, лише довший

i = 0
while i < len(tips_df):
    row = tips_df.iloc[i]                     # один рядок таблиці
    day = str(row["day"])
    time = str(row["time"])
    # симуляція офіціанта за зміною (день + час)
    if (day, time) == ("Thur", "Lunch") or (day, time) == ("Fri", "Lunch"):
        waiter = "Марія"
    elif (day, time) == ("Thur", "Dinner") or (day, time) == ("Sat", "Dinner"):
        waiter = "Тарас"
    else:                                     # Fri Dinner, Sun Dinner
        waiter = "Олексій"
    orders.append(Order(waiter, float(row["total_bill"]), float(row["tip"]), day, time, int(row["size"])))
    i += 1

print("Тип:", type(orders).__name__, " елемент:", type(orders[0]).__name__, " кількість:", len(orders))
print("Перший чек:", orders[0])


Головний висновок уроку: код, який ми написали для **5 вручну введених чеків**, працює **без жодної зміни** на 244 реальних. Алгоритм не знає і не піклується, звідки прийшли дані.


In [ ]:
# Той самий звіт, що й на 5 чеках — без змін у коді
revenue = 0
tips = 0
best = orders[0]
days = set()
waiters = set()

i = 0
while i < len(orders):
    order = orders[i]
    revenue += order.total_bill
    tips += order.tip
    if order.total_bill > best.total_bill:
        best = order
    days.add(order.day)
    waiters.add(order.waiter)
    i += 1

print(f"Чеків: {len(orders)}")
print(f"Виторг: ${revenue:.2f}, чайові: ${tips:.2f} ({tips / revenue * 100:.1f} %)")
print(f"Найбільший чек: ${best.total_bill} — {best.waiter}, {best.day} {best.time}, {best.size} гостей")
print(f"Днів: {len(days)} {sorted(days)}   офіціантів: {len(waiters)} {sorted(waiters)}")

assert len(orders) == 244
assert best.total_bill == 50.81
print("OK")


🔮 **Передбач:** власниця питає «а чайові кожного офіціанта окремо?». Який з наших трьох типів тут підійде?

<details><summary>Відповідь</summary>
Жоден. Накопичувач <code>tips</code> — одне число на всіх; множина <code>waiters</code> знає імена, але не вміє тримати число поруч з іменем. Потрібна структура «ім'я → число» — <b>словник</b>, тема уроку 6. Той самий <code>orders</code> там отримає рядок «чайові по офіціантах».
</details>


## 🔄 TRANSFER — задачі з перевірками

Той самий `orders: list[Order]` (244 чеки). Інструменти — тільки `list` / `tuple` / `set` і `while`; жодного `dict`. Кожна задача має `assert`: якщо він мовчить — розв'язок правильний.

### Задача 1 — аналітика вечері (Dinner)

Знайди всі чеки з `time == "Dinner"`, порахуй середній чек і середній % чайових серед них. Якщо таких чеків немає — поверни `(0, 0.0, 0.0)`, а не падай з `ZeroDivisionError`.


In [ ]:
def dinner_report(orders: list) -> tuple:
    """Повертає (кількість вечірніх чеків, середній чек, середній tip%)."""
    # YOUR CODE HERE
    pass


count, avg_bill, avg_tip_pct = dinner_report(orders)
print(f"Вечірніх чеків: {count}")
print(f"Середній чек:   ${avg_bill:.2f}")
print(f"Середній tip%:  {avg_tip_pct:.1f}%")

assert count == 176
assert round(avg_bill, 2) == 20.80
assert dinner_report([]) == (0, 0.0, 0.0)
print("OK")


### Задача 2 — великі столи

Знайди всі чеки, де `size >= 4`, порахуй середній чек і середній чек **на одну людину** (середнє значення `total_bill / size`). Для порожнього результату — `(0, 0.0, 0.0)`.


In [ ]:
def large_tables_report(orders: list) -> tuple:
    """Повертає (кількість великих столів, середній чек, середній чек на людину)."""
    # YOUR CODE HERE
    pass


count, avg_bill, avg_pp = large_tables_report(orders)
print(f"Великих столів (4+):    {count}")
print(f"Середній чек:           ${avg_bill:.2f}")
print(f"Середній чек на людину: ${avg_pp:.2f}")

assert count == 46
assert large_tables_report(orders[:2]) == (0, 0.0, 0.0)
print("OK")


### Задача 3 — хто працював у який день (множини)

Напиши `waiters_on(orders, day)`, що повертає **множину** імен офіціантів, які мають хоча б один чек у день `day`. Потім знайди, хто працював **і в четвер, і в п'ятницю** — однією операцією над множинами.


In [ ]:
def waiters_on(orders: list, day: str) -> set:
    """Множина офіціантів, що мають чеки в день day."""
    # YOUR CODE HERE
    pass


thu = waiters_on(orders, "Thur")
fri = waiters_on(orders, "Fri")
both = None   # ← заміни на операцію над множинами thu і fri
# YOUR CODE HERE

print("Четвер:  ", sorted(thu))
print("П'ятниця:", sorted(fri))
print("Обидва дні:", sorted(both))

assert thu == {"Марія", "Тарас"}
assert fri == {"Марія", "Олексій"}
assert both == {"Марія"}
assert waiters_on(orders, "Mon") == set()
print("OK")


### Задача 4 — агрегати без `max`/`min`/`sum`

Дано суми чеків за день. Циклом `while` (без вбудованих `max`, `min`, `sum`) знайди максимум, мінімум, середнє і кількість чеків вище середнього.


In [ ]:
bills = [540.0, 320.0, 980.0, 760.0, 450.0, 610.0, 1200.0, 275.0]

# YOUR CODE HERE  → змінні biggest, smallest, average, above
# YOUR CODE HERE

print(f"Максимум: {biggest}\nМінімум: {smallest}\nСереднє: {average:.1f}\nВище середнього: {above} чеків")

assert biggest == 1200.0 and smallest == 275.0
assert round(average, 1) == 641.9
assert above == 3
print("OK")


### Задача 5 — гості двох днів

Адміністраторка записувала гостей у список (із повторами). Знайди:
1. скільки **різних** гостей було в п'ятницю;
2. хто в п'ятницю приходив **більше одного разу** (множина);
3. хто був у п'ятницю, але **не** в суботу.


In [ ]:
friday_log   = ["Олена", "Богдан", "Марта", "Олена", "Ігор", "Марта", "Олена"]
saturday_log = ["Марта", "Ігор", "Софія", "Ігор"]

# YOUR CODE HERE  → unique_friday (int), repeat_guests (set), only_friday (set)
# YOUR CODE HERE

print("Різних гостей у пт:", unique_friday)
print("Приходили двічі+:  ", sorted(repeat_guests))
print("Лише в пт:         ", sorted(only_friday))

assert unique_friday == 4
assert repeat_guests == {"Олена", "Марта"}
assert only_friday == {"Олена", "Богдан"}
print("OK")


### Задача 6 — гості трьох днів

Знайди операціями над множинами:
1. гостей, що були **всі три** дні;
2. гостей, що були **рівно два** дні;
3. гостей, що були **лише** в п'ятницю.


In [ ]:
friday   = {"Олена", "Богдан", "Марта", "Ігор", "Софія"}
saturday = {"Богдан", "Ігор", "Петро", "Зоя", "Олена"}
sunday   = {"Марта", "Ігор", "Петро", "Ганна", "Богдан"}

# YOUR CODE HERE  → all_three, exactly_two, only_friday
# YOUR CODE HERE

print("Усі три дні:", sorted(all_three))
print("Рівно два:  ", sorted(exactly_two))
print("Лише пт:    ", sorted(only_friday))

assert all_three == {"Богдан", "Ігор"}
assert exactly_two == {"Олена", "Марта", "Петро"}
assert only_friday == {"Софія"}
print("OK")


### Задача 7* — сортування «бульбашкою»

Відсортуй суми чеків за зростанням двома вкладеними `while`: порівнюй сусідні пари і міняй місцями, якщо вони в неправильному порядку, доки список не стане відсортованим. Без `sorted()` і `.sort()`.


In [ ]:
bills = [640.0, 340.0, 250.0, 120.0, 220.0, 110.0, 900.0]
print("До:   ", bills)

# YOUR CODE HERE

print("Після:", bills)
assert bills == [110.0, 120.0, 220.0, 250.0, 340.0, 640.0, 900.0]
print("OK")


## ✅ Самоперевірка (6 запитань)

**1.** Що дасть `["a", "b", "c"][1:]`?

<details><summary>Відповідь</summary><code>['b', 'c']</code> — від індексу 1 до кінця.</details>

**2.** Після `a = [1]`, `b = a`, `b.append(2)` — що в `a`? А якщо `b = a.copy()`?

<details><summary>Відповідь</summary><code>[1, 2]</code>: обидва імені вказують на один список. З <code>.copy()</code> у <code>a</code> лишиться <code>[1]</code>.</details>

**3.** Чим `(5)` відрізняється від `(5,)`?

<details><summary>Відповідь</summary><code>(5)</code> — число 5, <code>(5,)</code> — кортеж з одного елемента. Кортеж створює кома.</details>

**4.** Чому для одного закритого чека обрали `tuple`/`NamedTuple`, а для всіх чеків дня — `list`?

<details><summary>Відповідь</summary>Чек — здійснений факт, кожна позиція має свій сенс, і його не повинні змінювати. Чеки дня — набір, що росте.</details>

**5.** `{}` — це порожній `set` чи порожній `dict`? Як отримати порожню множину?

<details><summary>Відповідь</summary><code>dict</code>. Порожня множина — лише <code>set()</code>.</details>

**6.** Чим `.remove()` відрізняється від `.discard()` у множини?

<details><summary>Відповідь</summary><code>.remove()</code> кидає <code>KeyError</code>, якщо елемента немає; <code>.discard()</code> мовчки нічого не робить.</details>


## Далі

Урок 6 («Словники, for, comprehensions») продовжує напряму з того самого `orders: list[Order]` — там цикл `while` з індексом стане коротшим завдяки `for`, а словник відповість на питання, яке лишилося без відповіді: чайові й виторг **окремо по кожному офіціанту і дню**.

- Книга: [Урок 5](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_05/) → [Урок 6](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_06/)
- Квест для груп: [cafe_shift_quest](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_05_lists_tuples_sets/cafe_shift_quest_student.ipynb)
